# PINN para reconstruir el campo de velocidades a partir de tres trayectorias

Adaptación directa del Problema 4 de `E1-TAPIA-ORDOÑEZ.ipynb`.

Se conserva la misma idea:

\[
\dot{\mathbf X}=\mathbf u_B(\mathbf X)
\]

con un campo de Burgers bidimensional y parámetros entrenables.

La diferencia es que ahora hay tres trayectorias experimentales. Cada trayectoria tiene su propia red
\(t\mapsto (x,y)\), pero las tres comparten los mismos parámetros físicos del vórtice.
También se permite aprender una pequeña corrección \((x_c,y_c)\) del centro del vórtice, porque los CSV
están referidos al centro del recipiente.


## Librerías y parámetros generales


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.manual_seed(1234)
np.random.seed(1234)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

plt.rcParams.update({
    "figure.figsize": (8, 7),
    "axes.grid": True,
    "font.family": "STIXGeneral",
    "mathtext.fontset": "cm",
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "legend.fontsize": 13,
})


## Cargar las tres trayectorias


In [ ]:
CARPETA = Path(
    r"C:\Users\user\Desktop\Labo5\1_Fluidos\datos\crudos"
)

ARCHIVOS = [
    CARPETA / "trayectoria1_PINN.csv",
    CARPETA / "trayectoria2_PINN.csv",
    CARPETA / "trayectoria3_PINN.csv",
]

dfs = [pd.read_csv(archivo) for archivo in ARCHIVOS]

for i, df in enumerate(dfs, start=1):
    print(
        f"Trayectoria {i}: "
        f"{len(df)} puntos, "
        f"T = {df['t_s'].max():.3f} s"
    )


## Visualización de los datos


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for i, df in enumerate(dfs, start=1):
    ax.plot(
        100 * df["x_m"],
        100 * df["y_m"],
        ".-",
        ms=2,
        lw=0.8,
        label=f"Trayectoria {i}"
    )

ax.set_xlabel("x [cm]")
ax.set_ylabel("y [cm]")
ax.set_title("Trayectorias experimentales")
ax.axis("equal")
ax.legend()
plt.show()


## Red neuronal

Como en el notebook original, cada trayectoria se representa mediante una MLP con activación `tanh`.

Como las trayectorias tienen duraciones muy distintas, se normaliza el tiempo de cada una a
\(\tau\in[-1,1]\). Esto evita que la `tanh` se sature para la trayectoria de aproximadamente 41 s.


In [ ]:
class MLP(torch.nn.Module):

    def __init__(self, sizes):
        super().__init__()

        self.layers = torch.nn.ModuleList()

        for i in range(len(sizes) - 1):
            self.layers.append(
                torch.nn.Linear(
                    sizes[i],
                    sizes[i + 1]
                )
            )

    def forward(self, x):

        h = x

        for hidden in self.layers[:-1]:
            h = torch.tanh(
                hidden(h)
            )

        return self.layers[-1](h)


## Parámetros físicos compartidos

El campo bidimensional usado en el Problema 4 del notebook es

\[
u_r=-\alpha r,
\qquad
u_\theta=
\frac{\Gamma}{2\pi r}
\left(
1-e^{-r^2/r_0^2}
\right).
\]

En cartesianas, tomando el centro del vórtice en \((x_c,y_c)\), se usa

\[
u_x=-\alpha x_r-y_r A,
\qquad
u_y=-\alpha y_r+x_r A,
\]

con

\[
A=
\frac{\Gamma}{2\pi r^2}
\left(
1-e^{-r^2/r_0^2}
\right).
\]

Los cinco parámetros \(\alpha,\Gamma,r_0,x_c,y_c\) son compartidos por las tres trayectorias.


In [ ]:
class ParametrosVortice(torch.nn.Module):

    def __init__(self):
        super().__init__()

        # Valores iniciales razonables en SI.
        # Son solamente condiciones iniciales del ajuste.

        self.alp = torch.nn.Parameter(
            torch.tensor([-0.05], dtype=torch.float32)
        )

        self.Gam = torch.nn.Parameter(
            torch.tensor([0.008], dtype=torch.float32)
        )

        self.r0 = torch.nn.Parameter(
            torch.tensor([0.010], dtype=torch.float32)
        )

        # Corrección inicial del centro: 0 m
        self.xc = torch.nn.Parameter(
            torch.tensor([0.0], dtype=torch.float32)
        )

        self.yc = torch.nn.Parameter(
            torch.tensor([0.0], dtype=torch.float32)
        )


parametros = ParametrosVortice().to(device)


## Campo de velocidades de Burgers


In [ ]:
def velocidad_burgers(X, parametros):

    x = X[:, 0]
    y = X[:, 1]

    xr = x - parametros.xc
    yr = y - parametros.yc

    r2 = xr**2 + yr**2 + 1e-12

    # r0 aparece al cuadrado; abs evita un radio negativo
    r0 = torch.abs(parametros.r0) + 1e-6

    comun = (
        parametros.Gam
        / (2 * torch.pi * r2)
        * (1 - torch.exp(-r2 / r0**2))
    )

    vx = (
        -parametros.alp * xr
        - yr * comun
    )

    vy = (
        -parametros.alp * yr
        + xr * comun
    )

    return torch.stack(
        (vx, vy),
        dim=1
    )


## Preparar los datos para Torch

Para no hacer innecesariamente pesado el entrenamiento, se puede usar un máximo de
`MAX_DATOS` puntos equiespaciados de cada trayectoria.

Si se quiere usar absolutamente todos los puntos, poner `MAX_DATOS = None`.


In [ ]:
MAX_DATOS = 500

datos_torch = []

for df in dfs:

    if MAX_DATOS is None or len(df) <= MAX_DATOS:
        indices = np.arange(len(df))
    else:
        indices = np.linspace(
            0,
            len(df) - 1,
            MAX_DATOS,
            dtype=int
        )

    t = torch.tensor(
        df["t_s"].values[indices],
        dtype=torch.float32,
        device=device
    ).view(-1, 1)

    X = torch.tensor(
        df[["x_m", "y_m"]].values[indices],
        dtype=torch.float32,
        device=device
    )

    T = float(
        df["t_s"].max()
    )

    # tau in [-1, 1]
    tau = (
        2 * t / T - 1
    )

    datos_torch.append(
        {
            "tau": tau,
            "X": X,
            "T": T,
        }
    )


## Entrenamiento de la PINN

La estructura de la pérdida es la misma que en el notebook:

\[
\mathcal L
=
MSE_d+\lambda MSE_f.
\]

El término de datos fuerza a cada MLP a pasar por su trayectoria experimental.

El término de física fuerza simultáneamente

\[
\frac{d\mathbf X_i}{dt}
=
\mathbf u_B(\mathbf X_i)
\]

para las tres trayectorias, usando un único conjunto de parámetros del campo.


In [ ]:
# Una MLP por trayectoria.
# Las tres comparten "parametros".

pinns = torch.nn.ModuleList([
    MLP([1, 40, 40, 40, 2]),
    MLP([1, 40, 40, 40, 2]),
    MLP([1, 40, 40, 40, 2]),
]).to(device)


optimizer = torch.optim.Adam(
    list(pinns.parameters())
    + list(parametros.parameters()),
    lr=2e-4
)


ITERATIONS = 20000
LAMBDA_FISICA = 1.0
N_FISICA = 300


# Puntos de física para cada trayectoria.
# Trabajamos en tau entre -1 y 1.

tau_physics = [
    torch.linspace(
        -1,
        1,
        N_FISICA,
        device=device
    ).view(-1, 1).requires_grad_(True)
    for _ in range(3)
]


loss_array = np.zeros(ITERATIONS)
error_datos_array = np.zeros(ITERATIONS)
error_fisica_array = np.zeros(ITERATIONS)


for epoch in range(ITERATIONS):

    optimizer.zero_grad()

    MSDd = 0.0
    MSDf = 0.0


    for pinn, datos, tau_f in zip(
        pinns,
        datos_torch,
        tau_physics
    ):

        # ====================================================
        # MSE de datos
        # ====================================================

        X_pred = pinn(
            datos["tau"]
        )

        MSDd_i = torch.mean(
            (X_pred - datos["X"])**2
        )

        MSDd = MSDd + MSDd_i


        # ====================================================
        # MSE de física
        # ====================================================

        X_f = pinn(
            tau_f
        )

        x_f = X_f[:, 0]
        y_f = X_f[:, 1]


        dx_dtau = torch.autograd.grad(
            x_f,
            tau_f,
            torch.ones_like(x_f),
            create_graph=True
        )[0]


        dy_dtau = torch.autograd.grad(
            y_f,
            tau_f,
            torch.ones_like(y_f),
            create_graph=True
        )[0]


        # tau = 2t/T - 1
        #
        # dX/dt = dX/dtau * 2/T

        factor_tiempo = (
            2.0 / datos["T"]
        )


        dX_dt = torch.cat(
            (
                dx_dtau * factor_tiempo,
                dy_dtau * factor_tiempo
            ),
            dim=1
        )


        velocidad = velocidad_burgers(
            X_f,
            parametros
        )


        residuo_fisica = (
            dX_dt - velocidad
        )


        MSDf_i = torch.mean(
            residuo_fisica**2
        )


        MSDf = MSDf + MSDf_i


    # Promedio sobre las tres trayectorias

    MSDd = MSDd / len(pinns)
    MSDf = MSDf / len(pinns)


    loss = (
        MSDd
        + LAMBDA_FISICA * MSDf
    )


    loss.backward()
    optimizer.step()


    loss_array[epoch] = (
        loss.detach().cpu().item()
    )

    error_datos_array[epoch] = (
        MSDd.detach().cpu().item()
    )

    error_fisica_array[epoch] = (
        MSDf.detach().cpu().item()
    )


    if epoch % 1000 == 0:

        print(
            f"Época {epoch:5d} | "
            f"MSE datos = {MSDd.item():.3e} | "
            f"MSE física = {MSDf.item():.3e} | "
            f"Loss = {loss.item():.3e}"
        )


## Evolución de la función de pérdida


In [ ]:
epocas = np.arange(
    ITERATIONS
)

plt.figure(figsize=(9, 7))

plt.plot(
    epocas,
    loss_array,
    label="Error total"
)

plt.plot(
    epocas,
    error_datos_array,
    label="Error datos"
)

plt.plot(
    epocas,
    LAMBDA_FISICA * error_fisica_array,
    label=r"$\lambda$ Error física"
)

plt.xlabel("N° de época")
plt.ylabel("Error")
plt.yscale("log")
plt.legend()
plt.show()


## Comparación entre las trayectorias experimentales y las aprendidas


In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 8)
)

for i, (pinn, df, datos) in enumerate(
    zip(pinns, dfs, datos_torch),
    start=1
):

    tau_eval = torch.linspace(
        -1,
        1,
        1000,
        device=device
    ).view(-1, 1)

    with torch.no_grad():

        X_estim = (
            pinn(tau_eval)
            .cpu()
            .numpy()
        )


    ax.plot(
        100 * df["x_m"],
        100 * df["y_m"],
        ".",
        ms=2,
        label=f"Datos {i}"
    )

    ax.plot(
        100 * X_estim[:, 0],
        100 * X_estim[:, 1],
        lw=2,
        label=f"PINN {i}"
    )


ax.set_xlabel("x [cm]")
ax.set_ylabel("y [cm]")
ax.set_title("Trayectorias: datos vs PINN")
ax.axis("equal")
ax.legend()
plt.show()


## Parámetros aprendidos


In [ ]:
alpha = parametros.alp.detach().cpu().item()
Gamma = parametros.Gam.detach().cpu().item()
r0 = abs(
    parametros.r0.detach().cpu().item()
)

xc = parametros.xc.detach().cpu().item()
yc = parametros.yc.detach().cpu().item()


print(f"alpha = {alpha:.6g} 1/s")
print(f"Gamma = {Gamma:.6g} m²/s")
print(f"r0    = {r0:.6g} m")
print(f"xc    = {xc:.6g} m")
print(f"yc    = {yc:.6g} m")


## Campo de velocidades reconstruido

Una vez aprendidos los parámetros, el campo ya no depende de las redes de trayectoria:
se evalúa directamente el modelo de Burgers con los parámetros estimados.

El gráfico se limita aproximadamente a la región recorrida por las partículas para evitar interpretar
como datos experimentales una extrapolación muy lejana.


In [ ]:
# Región cubierta por los datos, con un pequeño margen

todos_x = np.concatenate(
    [df["x_m"].values for df in dfs]
)

todos_y = np.concatenate(
    [df["y_m"].values for df in dfs]
)

margen = 0.005

xmin = todos_x.min() - margen
xmax = todos_x.max() + margen

ymin = todos_y.min() - margen
ymax = todos_y.max() + margen


x_grid = np.linspace(
    xmin,
    xmax,
    31
)

y_grid = np.linspace(
    ymin,
    ymax,
    31
)

Xg, Yg = np.meshgrid(
    x_grid,
    y_grid
)


XY = torch.tensor(
    np.column_stack(
        (
            Xg.ravel(),
            Yg.ravel()
        )
    ),
    dtype=torch.float32,
    device=device
)


with torch.no_grad():

    UV = (
        velocidad_burgers(
            XY,
            parametros
        )
        .cpu()
        .numpy()
    )


Ug = UV[:, 0].reshape(
    Xg.shape
)

Vg = UV[:, 1].reshape(
    Yg.shape
)

modulo = np.sqrt(
    Ug**2 + Vg**2
)


fig, ax = plt.subplots(
    figsize=(9, 8)
)


q = ax.quiver(
    100 * Xg,
    100 * Yg,
    Ug,
    Vg,
    modulo
)


for i, df in enumerate(
    dfs,
    start=1
):

    ax.plot(
        100 * df["x_m"],
        100 * df["y_m"],
        lw=1.5,
        label=f"Trayectoria {i}"
    )


ax.scatter(
    [100 * xc],
    [100 * yc],
    marker="x",
    s=100,
    label="Centro aprendido"
)


ax.set_xlabel("x [cm]")
ax.set_ylabel("y [cm]")
ax.set_title("Campo de velocidades reconstruido")
ax.axis("equal")
ax.legend()

cbar = plt.colorbar(
    q,
    ax=ax
)

cbar.set_label(
    "|u| [m/s]"
)

plt.show()


## Perfiles radial y tangencial del campo aprendido


In [ ]:
r_max = np.max(
    np.sqrt(
        (todos_x - xc)**2
        + (todos_y - yc)**2
    )
)

r = np.linspace(
    1e-5,
    r_max,
    500
)


u_r = (
    -alpha * r
)


u_theta = (
    Gamma
    / (2 * np.pi * r)
    * (
        1
        - np.exp(
            -r**2 / r0**2
        )
    )
)


plt.figure(
    figsize=(9, 7)
)

plt.plot(
    100 * r,
    100 * u_r,
    label=r"$u_r$"
)

plt.plot(
    100 * r,
    100 * u_theta,
    label=r"$u_\theta$"
)

plt.xlabel("r [cm]")
plt.ylabel("Velocidad [cm/s]")
plt.title("Perfiles del campo aprendido")
plt.legend()
plt.show()
